<a href="https://colab.research.google.com/github/MilkySon/RL_carracing_competition/blob/261106-beta-ppo/BetaPPO_Temporal96_Skip2_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CarRacing-v3 · Temporal BetaPPO · 96×96 · Skip 2 · 3-step TD

Colab에서 GPU 런타임을 선택하고 위에서부터 실행합니다. 공식 채점기와 같은 RGB 입력을
받아 96×96 휘도로 바꾸고, 최근 4장의 시간축과 공간축을 Conv3d로 함께 처리합니다.
행동은 원본 2프레임 동안 반복됩니다. 관측 스택은 보통 t−6, t−4, t−2, t를 봅니다.

기존 Gray84 실험과 동일하게 actor와 critic에 3-step TD를 쓰고, 학습용 보상 배율 0.1과
signed pedal 변환을 유지합니다. 3-step은 최대 6 원본 프레임이며, 기존 skip 4에서는
최대 12 프레임이었습니다. 따라서 이전 버전과의 비교에는 추정 시간 범위의 차이도 포함됩니다.
이탈 후에도 원본 환경의 종료 또는 시간 제한까지 주행합니다.

두 전문가 구성에서는 화면으로부터 gate가 매 결정 시점의 혼합 가중치를 학습합니다.
도로·잔디 레이블은 분석에만 쓰므로 전문가의 특화 여부는 결과로 확인합니다.
학습 시 혼합 Beta에서 행동을 샘플링하고, 제출 시 잠재 조향·페달의 혼합 평균을 변환합니다.
gate의 전환은 2프레임마다 가능하며, 전문가별 행동 반복 길이는 고정입니다.
같은 설정의 전문가 1개와 2개를 비교하면 전문가를 추가한 효과를 분리할 수 있습니다.

## 1. 설치 및 실험 설정

In [ ]:
%pip -q install gymnasium==1.2.2 box2d==2.3.10 pygame==2.6.1 onnx==1.23.0 onnxruntime==1.23.2

In [ ]:
from datetime import datetime
from pathlib import Path
import os

os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
TEAM_NAME = "Reinforcement_material"                 # 제출 전에 실제 팀 이름으로 수정
TRAIN_SEED = 42
EXPERT_COUNTS = (1, 2)                # 먼저 한 번만 돌리려면 (1,)
BUDGET = 300_000                     # 모델별 원본 환경 학습 프레임. 빠른 점검은 128
VALIDATION_SEEDS = tuple(range(500, 505))
EVAL_SEEDS = tuple(range(1000, 1020))
DIAGNOSTIC_SEEDS = (1004, 1007)
SAMPLE_REPEATS = 3

# 세션 종료 후에도 남기려면 Drive를 마운트하고 아래 경로를 Drive 아래로 변경.
from google.colab import drive
drive.mount("/content/drive")
RUN_BASE = Path("temporal_betappo_runs") / datetime.now().strftime("%Y%m%d_%H%M%S")
# 평가만 다시 하려면 {1: "경로/best.pt", 2: "경로/best.pt"} 형태로 지정.
EXISTING_CHECKPOINTS = {}

Mounted at /content/drive


## 2. 공식 채점기와 실험 코드
채점기는 제공된 대회 ZIP의 원본 파일입니다.

In [ ]:
%%writefile check_submission.py
"""공식 채점기 (ONNX 제출 파일용).  [수정 금지]

교수자 채점 서버도 이 파일을 그대로 사용합니다. 제출 전에 반드시 직접 돌려 보세요.
    python check_submission.py --onnx MyTeam.onnx
    python check_submission.py --onnx MyTeam.onnx --seeds 1000-1019

채점 방식:
    - CarRacing-v3 원본 환경 + HideScoreText(화면 아래 점수 숫자를 그리지 않음) (이 저장소의 envs.py는 사용하지 않음)
    - 최근 n_stack개 agent step의 원본 RGB 프레임 (1, n_stack, 96, 96, 3) uint8을 정책에 입력
      (에피소드 시작 시에는 첫 프레임을 n_stack번 반복)
    - 정책 출력 행동을 frame_skip 프레임 동안 반복
    - 점수 = 트랙(seed)별 원본 보상 return 평균

ONNX 파일 규격 (export.py가 자동으로 맞춰 줌):
    입력 "frames" (1, n_stack, 96, 96, 3) uint8
    출력 "action" 이산: (1,) int64 in [0, 4] / 연속: (1, 3) float32 (범위 밖 값은 잘라냄)
    metadata: continuous (true/false), frame_skip (1~8), n_stack (1~16)
"""

import argparse
import json
import os
import sys
from collections import deque

import gymnasium as gym
import numpy as np
import onnx
import onnxruntime as ort
import pygame

PUBLIC_EVAL_SEEDS = list(range(1000, 1010))
MAX_FILE_MB = 100


class HideScoreText(gym.Wrapper):
    """화면 아래의 점수 숫자를 그리지 않는다. [수정 금지]

    숫자 글꼴은 pygame 버전과 운영체제마다 다르게 그려져서, 같은 정책도 컴퓨터마다 점수가 크게 달라지는
    원인이었다. 숫자를 빼면 트랙, 차, 계기판 막대는 어느 컴퓨터에서나 같은 화면이 된다.
    학습 환경(envs.py)과 공식 채점기(check_submission.py)가 같은 코드를 쓴다."""

    class _NoFont:
        def __init__(self, *args, **kwargs):
            pass

        def render(self, *args, **kwargs):
            return pygame.Surface((0, 0))

    def _without_font(self, fn, *args, **kwargs):
        original = pygame.font.Font
        pygame.font.Font = self._NoFont
        try:
            return fn(*args, **kwargs)
        finally:
            pygame.font.Font = original

    def reset(self, **kwargs):
        return self._without_font(self.env.reset, **kwargs)

    def step(self, action):
        return self._without_font(self.env.step, action)

    def render(self):
        return self._without_font(self.env.render)


class SubmissionError(Exception):
    pass


def load_policy(path):
    if os.path.getsize(path) > MAX_FILE_MB * 1024 * 1024:
        raise SubmissionError(f"파일이 {MAX_FILE_MB}MB를 넘습니다. (File is larger than {MAX_FILE_MB}MB.)")
    try:
        model = onnx.load(path)
    except Exception as e:
        raise SubmissionError(f"ONNX 파일을 읽을 수 없습니다. (Cannot read the ONNX file.) {e}")
    meta = {p.key: p.value for p in model.metadata_props}
    try:
        cfg = {
            "continuous": meta["continuous"].lower() == "true",
            "frame_skip": int(meta["frame_skip"]),
            "n_stack": int(meta["n_stack"]),
        }
    except (KeyError, ValueError):
        raise SubmissionError("metadata(continuous, frame_skip, n_stack)가 없습니다. export.py로 만드세요. "
                              "(Missing metadata. Create the file with export.py.)")
    if not 1 <= cfg["frame_skip"] <= 8 or not 1 <= cfg["n_stack"] <= 16:
        raise SubmissionError(f"허용 범위를 벗어난 metadata (Metadata out of the allowed range): {cfg}")

    opts = ort.SessionOptions()
    opts.intra_op_num_threads = 1
    opts.inter_op_num_threads = 1
    # 경로 대신 바이트로 전달: 일부 유니코드 파일 이름(예: macOS 한글)을 onnxruntime이 경로로 열지 못함
    sess = ort.InferenceSession(model.SerializeToString(), opts, providers=["CPUExecutionProvider"])
    inputs = [i.name for i in sess.get_inputs()]
    outputs = [o.name for o in sess.get_outputs()]
    if inputs != ["frames"] or "action" not in outputs:
        raise SubmissionError(f"입력은 'frames', 출력은 'action'이어야 합니다 (Input must be 'frames' and output 'action'): "
                              f"{inputs} -> {outputs}")
    return sess, cfg


def to_env_action(raw, cfg, env):
    a = np.asarray(raw).reshape(-1)
    if cfg["continuous"]:
        if a.shape != (3,) or not np.all(np.isfinite(a)):
            raise SubmissionError(f"연속 행동은 유한한 값 3개여야 합니다 (A continuous action must be 3 finite values): {a}")
        return np.clip(a.astype(np.float32), env.action_space.low, env.action_space.high)
    if a.shape != (1,) or not 0 <= int(a[0]) < env.action_space.n:
        raise SubmissionError(f"이산 행동은 0~{env.action_space.n - 1} 정수 하나여야 합니다 "
                              f"(A discrete action must be a single integer in 0~{env.action_space.n - 1}): {a}")
    return int(a[0])


def run_episode(sess, cfg, seed):
    env = HideScoreText(gym.make("CarRacing-v3", continuous=cfg["continuous"]))
    obs, _ = env.reset(seed=seed)
    frames = deque([obs] * cfg["n_stack"], maxlen=cfg["n_stack"])
    total, done = 0.0, False
    while not done:
        x = np.stack(frames)[None]  # (1, n_stack, 96, 96, 3) uint8
        action = to_env_action(sess.run(["action"], {"frames": x})[0], cfg, env)
        for _ in range(cfg["frame_skip"]):
            obs, reward, terminated, truncated, _ = env.step(action)
            total += reward
            done = terminated or truncated
            if done:
                break
        frames.append(obs)
    env.close()
    return total


def evaluate_onnx(path, seeds):
    sess, cfg = load_policy(path)
    return cfg, [run_episode(sess, cfg, s) for s in seeds]


def parse_seeds(s):
    if "-" in s:
        a, b = map(int, s.split("-"))
        return list(range(a, b + 1))
    return [int(x) for x in s.split(",")]


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--onnx", required=True)
    p.add_argument("--seeds", type=parse_seeds, default=PUBLIC_EVAL_SEEDS)
    p.add_argument("--json", default=None, help="결과 JSON 저장 경로 (채점 서버용)")
    p.add_argument("--quiet", action="store_true")
    args = p.parse_args()

    try:
        cfg, returns = evaluate_onnx(args.onnx, args.seeds)
        result = {"ok": True, "config": cfg, "returns": returns,
                  "mean": float(np.mean(returns)), "std": float(np.std(returns))}
    except SubmissionError as e:
        result = {"ok": False, "error": str(e)}
    except Exception as e:  # 실행 중 오류 (shape 불일치 등)
        result = {"ok": False, "error": f"{type(e).__name__}: {e}"[:500]}

    if args.json:
        with open(args.json, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False)
    if not args.quiet:
        if result["ok"]:
            print(f"설정: {result['config']}")
            if len(args.seeds) <= 20:
                for s, r in zip(args.seeds, result["returns"]):
                    print(f"  seed {s}: {r:8.1f}")
            print(f"SCORE: {result['mean']:.1f} ± {result['std']:.1f}  ({len(args.seeds)} tracks)")
        else:
            print(f"제출 파일 오류: {result['error']}")
    sys.exit(0 if result["ok"] else 1)

Writing check_submission.py


In [ ]:
%%writefile temporal_betappo.py
from collections import deque
from pathlib import Path
import csv
import json
import os
import random

import gymnasium as gym
import numpy as np
import onnx
import torch
from torch import nn
from torch.nn import functional as F
from torch.distributions import Beta, Categorical

from check_submission import HideScoreText, load_policy, run_episode, to_env_action

N_STACK = 4
FRAME_SKIP = 2
TD_STEPS = 3
GAMMA = 0.99
REWARD_SCALE = 0.1
VALIDATION_SEEDS = tuple(range(500, 505))


def controls(unit_action):
    steer, pedal = (2 * unit_action - 1).unbind(-1)
    return torch.stack((steer, pedal.relu(), (-pedal).relu()), dim=-1)


class TemporalBetaPolicy(nn.Module):
    def __init__(self, experts):
        super().__init__()
        if experts not in (1, 2):
            raise ValueError("Choose one or two experts")
        self.experts = experts
        self.encoder = nn.Sequential(
            nn.Conv3d(1, 16, (2, 8, 8), stride=(1, 4, 4)), nn.ReLU(),
            nn.Conv3d(16, 32, (3, 4, 4), stride=(1, 2, 2)), nn.ReLU(),
            nn.Conv3d(32, 48, (1, 3, 3), stride=(1, 2, 2)), nn.ReLU(),
            nn.Flatten(), nn.Linear(48 * 4 * 4, 256), nn.ReLU(),
        )
        self.actor = nn.Linear(256, 4 * experts)
        self.critic = nn.Linear(256, 1)
        self.progress = nn.Linear(258, 1)
        self.gate = nn.Linear(256, experts) if experts > 1 else None
        nn.init.orthogonal_(self.actor.weight, gain=0.01)
        with torch.no_grad():
            self.actor.bias.copy_(torch.tensor([0., 1.5, 0., -0.5]).repeat(experts))
        if self.gate is not None:
            nn.init.zeros_(self.gate.weight)
            nn.init.zeros_(self.gate.bias)

    def parameters_at(self, frames):
        rgb = frames.float() / 255.
        gray = 0.299 * rgb[..., 0] + 0.587 * rgb[..., 1] + 0.114 * rgb[..., 2]
        features = self.encoder(gray.unsqueeze(1))
        alpha, beta = (1 + F.softplus(self.actor(features).reshape(-1, self.experts, 4))).chunk(2, -1)
        logits = self.gate(features) if self.gate is not None else alpha[:, :, 0] * 0
        value = self.critic(features).squeeze(-1)
        return alpha, beta, logits, value, features

    def forward(self, frames):
        alpha, beta, logits, _, _ = self.parameters_at(frames)
        mean_unit = (logits.softmax(-1).unsqueeze(-1) * (alpha / (alpha + beta))).sum(-2)
        return controls(mean_unit)


def log_prob(alpha, beta, logits, actions):
    component_log_prob = Beta(alpha, beta).log_prob(actions.unsqueeze(-2)).sum(-1)
    return torch.logsumexp(logits.log_softmax(-1) + component_log_prob, dim=-1)


def sample_actions(alpha, beta, logits, count=1):
    components = Categorical(logits=logits).sample((count,))
    index = components.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, 1, 2)
    chosen_alpha = alpha.unsqueeze(0).expand(count, -1, -1, -1).gather(2, index).squeeze(2)
    chosen_beta = beta.unsqueeze(0).expand(count, -1, -1, -1).gather(2, index).squeeze(2)
    return Beta(chosen_alpha, chosen_beta).sample().clamp(1e-6, 1 - 1e-6), components


class Track:
    def __init__(self):
        self.env = HideScoreText(gym.make("CarRacing-v3", continuous=True))
        self.frames = deque(maxlen=N_STACK)

    def reset(self, seed):
        obs, _ = self.env.reset(seed=seed)
        self.frames.extend([obs] * N_STACK)
        return np.stack(self.frames)

    def step(self, action, limit=FRAME_SKIP, inspect=False):
        reward = 0.
        trace = []
        for count in range(1, limit + 1):
            obs, r, terminated, truncated, _ = self.env.step(action)
            reward += r
            if inspect:
                car = self.env.unwrapped.car
                trace.append((sum(bool(w.tiles) for w in car.wheels),
                              self.env.unwrapped.tile_visited_count,
                              float(car.hull.linearVelocity.length),
                              float(car.hull.angularVelocity)))
            if terminated or truncated:
                break
        self.frames.append(obs)
        return np.stack(self.frames), reward, terminated, truncated, count, trace

    def close(self):
        self.env.close()


def advantages(rewards, values, next_values, terminated, truncated):
    """Time first, lanes second. Each lane ends independently."""
    delta = REWARD_SCALE * rewards + GAMMA * np.where(terminated, 0., next_values) - values
    result = delta.copy()
    alive = ~(terminated | truncated)
    continuation = alive.copy()
    for offset in range(1, TD_STEPS):
        result[:-offset] += GAMMA ** offset * continuation[:-offset] * delta[offset:]
        continuation[:-offset] &= alive[offset:]
    return result


def tensor(array, model):
    return torch.as_tensor(array, device=next(model.parameters()).device)


def write_csv(path, rows):
    if rows:
        with Path(path).open("w", newline="") as file:
            writer = csv.DictWriter(file, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)


def save_checkpoint(path, model, raw_frames, seed):
    temporary = path.with_suffix(".tmp")
    torch.save(dict(experts=model.experts, weights=model.state_dict(),
                    raw_frames=raw_frames, seed=seed), temporary)
    temporary.replace(path)


def load_checkpoint(path, device="cpu"):
    data = torch.load(path, map_location=device, weights_only=True)
    model = TemporalBetaPolicy(data["experts"]).to(device)
    model.load_state_dict(data["weights"])
    return model.eval()


def update(model, optimizer, records):
    obs, unit, old_log, values, next_values, rewards, term, trunc, valid = map(np.stack, zip(*records))
    adv = advantages(rewards, values, next_values, term, trunc)
    valid = valid.ravel()
    target = tensor((adv + values).astype(np.float32).ravel()[valid], model)
    adv = tensor(adv.astype(np.float32).ravel()[valid], model)
    adv = (adv - adv.mean()) / (adv.std(unbiased=False) + 1e-8)
    obs = obs.reshape(-1, N_STACK, 96, 96, 3)[valid]
    unit = tensor(unit.reshape(-1, 2)[valid], model)
    old_log = tensor(old_log.ravel()[valid], model)
    gain = tensor((rewards > 0).ravel()[valid].astype(np.float32), model)
    for _ in range(4):
        divergences = []
        for idx_cpu in torch.randperm(len(obs)).split(64):
            idx = idx_cpu.to(adv.device)
            a, b, logits, pred, features = model.parameters_at(tensor(obs[idx_cpu.numpy()], model))
            log_ratio = log_prob(a, b, logits, unit[idx]) - old_log[idx]
            ratio = log_ratio.exp()
            policy_loss = -torch.minimum(ratio * adv[idx], ratio.clamp(.8, 1.2) * adv[idx]).mean()
            progress = model.progress(torch.cat((features, unit[idx]), dim=-1)).squeeze(-1)
            # Joint gate/component entropy is a regularizer. The PPO ratio uses marginal density.
            gate = Categorical(logits=logits)
            entropy = gate.entropy() + (gate.probs * Beta(a, b).entropy().sum(-1)).sum(-1)
            loss = (policy_loss + 0.5 * F.mse_loss(pred, target[idx])
                    + .05 * F.binary_cross_entropy_with_logits(progress, gain[idx])
                    - .005 * entropy.mean())
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), .5, error_if_nonfinite=True)
            optimizer.step()
            divergences.append((ratio.detach() - 1 - log_ratio.detach()).mean().item())
        kl = float(np.mean(divergences))
        if kl > .025:
            break
    return kl


@torch.no_grad()
def mean_score(model, seed):
    track = Track()
    try:
        obs = track.reset(seed)
        total = 0.
        while True:
            action = model(tensor(obs[None], model))[0].cpu().numpy()
            obs, reward, term, trunc, _, _ = track.step(action)
            total += reward
            if term or trunc:
                return total
    finally:
        track.close()


def train(root, experts, seed=42, budget=300_000, validation_seeds=VALIDATION_SEEDS):
    if budget % 8 or not 8 <= budget <= 300_000:
        raise ValueError("Budget must be a multiple of 8 between 8 and 300,000")
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    if any(root.iterdir()):
        raise ValueError(f"Choose an empty run directory: {root}")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = TemporalBetaPolicy(experts).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=.00025, eps=1e-5)
    config = dict(experts=experts, seed=seed, budget=budget, frame_skip=FRAME_SKIP,
                  n_stack=N_STACK, image_size=96, td_steps=TD_STEPS, gamma=GAMMA,
                  reward_scale=REWARD_SCALE, rollout=128, lanes=4)
    (root / "config.json").write_text(json.dumps(config, indent=2))
    tracks = [Track() for _ in range(4)]
    episodes = np.zeros(4, dtype=int)
    scores = np.zeros(4)
    counts = np.zeros(4, dtype=int)
    training_rows, validation_rows = [], []
    best = -np.inf
    quota = budget // len(tracks)
    next_eval = 25_000

    def reset(lane):
        return tracks[lane].reset(100_000 * (seed + 1) + 4 * int(episodes[lane]) + lane)

    try:
        obs = np.stack([reset(i) for i in range(len(tracks))])
        while np.any(counts < quota):
            records = []
            for _ in range(128):
                valid = counts < quota
                if not valid.any():
                    break
                with torch.no_grad():
                    a, b, logits, values, _ = model.parameters_at(tensor(obs, model))
                    unit = sample_actions(a, b, logits)[0][0]
                    old_log = log_prob(a, b, logits, unit)
                    action = controls(unit).cpu().numpy()
                steps = [track.step(u, min(FRAME_SKIP, quota - int(counts[i])))
                         if valid[i] else (obs[i], 0., True, False, 0, [])
                         for i, (track, u) in enumerate(zip(tracks, action))]
                successors, rewards, term, trunc, consumed, _ = zip(*steps)
                with torch.no_grad():
                    next_values = model.parameters_at(tensor(np.stack(successors), model))[3]
                records.append((obs, unit.cpu().numpy(), old_log.cpu().numpy(),
                                values.cpu().numpy(), next_values.cpu().numpy(),
                                rewards, term, np.asarray(trunc) | (counts + consumed == quota), valid))
                counts += consumed
                scores += rewards
                next_obs = list(successors)
                for i, (terminated, truncated) in enumerate(zip(term, trunc)):
                    if valid[i] and (terminated or truncated):
                        training_rows.append(dict(raw_frames=int(counts.sum()), lane=i,
                                                  score=float(scores[i])))
                        episodes[i] += 1
                        scores[i] = 0.
                        if counts[i] < quota:
                            next_obs[i] = reset(i)
                obs = np.stack(next_obs)
            kl = update(model, optimizer, records)
            used = int(counts.sum())
            if used >= next_eval or used == budget:
                returns = [mean_score(model, track_seed) for track_seed in validation_seeds]
                score = float(np.mean(returns))
                validation_rows.append(dict(raw_frames=used, mean=score,
                                            std=float(np.std(returns)), kl=kl))
                write_csv(root / "train.csv", training_rows)
                write_csv(root / "validation.csv", validation_rows)
                save_checkpoint(root / "latest.pt", model, used, seed)
                if score > best:
                    best = score
                    save_checkpoint(root / "best.pt", model, used, seed)
                print(f"{used:,}/{budget:,} raw frames | validation {score:.1f} | KL {kl:.4f}",
                      flush=True)
                next_eval = (used // 25_000 + 1) * 25_000
        save_checkpoint(root / "final.pt", model, budget, seed)
        return root / "best.pt"
    finally:
        for track in tracks:
            track.close()


def assert_action_mapping():
    unit = torch.tensor([[0., 0.], [.5, .5], [1., 1.], [.25, .25], [.75, .75]])
    expected = [[-1., 0., 1.], [0., 0., 0.], [1., 1., 0.],
                [-.5, 0., .5], [.5, .5, 0.]]
    torch.testing.assert_close(controls(unit), torch.tensor(expected))


@torch.no_grad()
def export_and_verify(model, path, seed=1004):
    assert_action_mapping()
    model = model.to("cpu").eval()
    dummy = torch.zeros((1, N_STACK, 96, 96, 3), dtype=torch.uint8)
    torch.onnx.export(model, (dummy,), str(path), input_names=["frames"],
                      output_names=["action"], opset_version=17, dynamo=False)
    graph = onnx.load(path)
    for key, value in {"continuous": "true", "frame_skip": str(FRAME_SKIP),
                       "n_stack": str(N_STACK)}.items():
        graph.metadata_props.add(key=key, value=value)
    onnx.checker.check_model(graph)
    onnx.save(graph, path)
    session, config = load_policy(path)
    track = Track()
    try:
        obs = track.reset(seed)
        decisions, max_error = 0, 0.
        while True:
            frames = obs[None]
            torch_action = model(torch.from_numpy(frames))[0].numpy()
            raw = session.run(["action"], {"frames": frames})[0]
            action = to_env_action(raw, config, track.env)
            np.testing.assert_array_equal(raw[0], action)  # Official clipping must be a no-op.
            np.testing.assert_allclose(action, torch_action, atol=1e-5, rtol=1e-5)
            max_error = max(max_error, float(np.max(np.abs(action - torch_action))))
            decisions += 1
            obs, _, term, trunc, _, _ = track.step(action)
            if term or trunc:
                break
        print(f"ONNX parity | {decisions} decisions | maximum error {max_error:.2e}")
    finally:
        track.close()
    return session, config


def road_phase(track):
    env = track.env.unwrapped
    wheels = sum(bool(w.tiles) for w in env.car.wheels)
    if wheels == 0:
        return "grass"
    if wheels < 4:
        return "edge"
    points = np.asarray(env.track)
    xy = np.asarray(env.car.hull.position)
    nearest = np.argmin(((points[:, 2:4] - xy) ** 2).sum(-1))
    angle = points[(nearest + 5) % len(points), 1] - points[nearest, 1]
    bend = abs(np.arctan2(np.sin(angle), np.cos(angle)))
    return "curve" if bend >= .35 else "road"


@torch.no_grad()
def action_summary(alpha, beta, logits, samples=64):
    probs = logits.softmax(-1)
    heads = alpha / (alpha + beta)
    mean_unit = (probs.unsqueeze(-1) * heads).sum(-2)
    mean_action = controls(mean_unit)[0].numpy()
    draws, _ = sample_actions(alpha, beta, logits, samples)
    env_draws = controls(draws[:, 0]).numpy()
    quantiles = np.quantile(env_draws, [.05, .50, .95], axis=0)
    row = dict(mean_pedal_unit=float(mean_unit[0, 1]),
               sample_left=float((env_draws[:, 0] < -.05).mean()),
               sample_right=float((env_draws[:, 0] > .05).mean()),
               sample_braking=float((env_draws[:, 2] > .05).mean()),
               sample_accelerating=float((env_draws[:, 1] > .05).mean()),
               sample_both=float(((env_draws[:, 1] > 0) & (env_draws[:, 2] > 0)).mean()))
    for col, name in enumerate(("steer", "gas", "brake")):
        row[f"mean_{name}"] = float(mean_action[col])
        row[f"sample_mean_{name}"] = float(env_draws[:, col].mean())
        for label, q in zip(("p05", "p50", "p95"), quantiles[:, col]):
            row[f"sample_{name}_{label}"] = float(q)
    for expert, weight in enumerate(probs[0].tolist()):
        row[f"gate_{expert}"] = weight
        for col, name in enumerate(("steer", "gas", "brake")):
            row[f"expert_{expert}_{name}"] = float(controls(heads)[0, expert, col])
    return mean_action, row


@torch.no_grad()
def diagnose_episode(model, seed, sampled, replicate, log_actions):
    # Isolate diagnostic draws from the RNG that chooses the driven actions.
    torch.manual_seed(500_000 + seed * 101 + replicate)
    track = Track()
    actions, exits = [], []
    pending = None
    frames = stable = 0
    score = 0.
    try:
        obs = track.reset(seed)
        wheels = sum(bool(w.tiles) for w in track.env.unwrapped.car.wheels)
        # Before the first physics step, a reset may have no wheel contact yet.
        seen_road = wheels > 0
        while True:
            phase = road_phase(track)
            a, b, logits, _, _ = model.parameters_at(torch.from_numpy(obs[None]))
            if sampled:
                draw, _ = sample_actions(a, b, logits)
                action = controls(draw[0, 0]).numpy()
            else:
                action = model(torch.from_numpy(obs[None]))[0].numpy()
            if log_actions:
                with torch.random.fork_rng(devices=[]):
                    torch.manual_seed(10_000 + seed * 101 + frames)
                    expected, summary = action_summary(a, b, logits)
                car = track.env.unwrapped.car
                clipped = to_env_action(action, {"continuous": True}, track.env)
                np.testing.assert_array_equal(clipped, action)
                np.testing.assert_allclose(action, expected, atol=1e-6)
                actions.append(dict(seed=seed, frame=frames, phase=phase,
                                    wheels_on_road=wheels, speed=float(car.hull.linearVelocity.length),
                                    yaw_rate=float(car.hull.angularVelocity),
                                    actual_steer=float(clipped[0]), actual_gas=float(clipped[1]),
                                    actual_brake=float(clipped[2]), **summary))
            obs, reward, term, trunc, _, trace = track.step(action, inspect=True)
            score += reward
            for wheels, tiles, _, _ in trace:
                frames += 1
                if wheels == 0 and seen_road and pending is None:
                    pending = dict(start=frames, touch=None, stable=None, progress=None,
                                   tiles_at_exit=tiles, tiles_at_touch=None)
                    stable = 0
                if wheels > 0:
                    seen_road = True
                if pending is None:
                    continue
                if wheels > 0:
                    if pending["touch"] is None:
                        pending["touch"] = frames - pending["start"]
                        pending["tiles_at_touch"] = tiles
                    stable += 1
                    if stable >= 10 and pending["stable"] is None:
                        pending["stable"] = frames - pending["start"]
                    if tiles > pending["tiles_at_exit"] and pending["progress"] is None:
                        pending["progress"] = frames - pending["start"]
                else:
                    stable = 0
                if (pending["stable"] is not None and pending["progress"] is not None
                    or frames - pending["start"] >= 100):
                    exits.append(pending)
                    pending = None
                    seen_road = wheels > 0
            if term or trunc:
                if pending is not None:
                    exits.append(pending)
                break
        row = dict(seed=seed, mode="sample" if sampled else "mean", replicate=replicate,
                   score=score, raw_frames=frames, exits=len(exits),
                   recontacts=sum(e["touch"] is not None for e in exits),
                   stable_recoveries=sum(e["stable"] is not None for e in exits),
                   progressed=sum(e["progress"] is not None for e in exits))
        exit_rows = [dict(seed=seed, mode=row["mode"], replicate=replicate, **e) for e in exits]
        return row, actions, exit_rows
    finally:
        track.close()


def evaluate_and_diagnose(model, root, session, config, seeds=range(1000, 1020),
                          diagnostic_seeds=(1004, 1007), sample_repeats=3):
    model = model.to("cpu").eval()
    root = Path(root)
    official = [dict(seed=seed, score=run_episode(session, config, seed)) for seed in seeds]
    write_csv(root / "official_scores.csv", official)
    scores, actions, exits = [], [], []
    for seed in diagnostic_seeds:
        for replicate in range(sample_repeats + 1):
            row, action_rows, exit_rows = diagnose_episode(
                model, seed, sampled=replicate > 0, replicate=replicate, log_actions=replicate == 0)
            scores.append(row)
            actions.extend(action_rows)
            exits.extend(exit_rows)
    write_csv(root / "diagnostic_scores.csv", scores)
    write_csv(root / "action_diagnostics.csv", actions)
    write_csv(root / "recovery_events.csv", exits)
    official_scores = [row["score"] for row in official]
    print(f"Official ONNX | {len(official_scores)} tracks | "
          f"{np.mean(official_scores):.2f} ± {np.std(official_scores):.2f}")
    return scores

Writing temporal_betappo.py


## 3. 학습 및 체크포인트 선택
학습 중에는 500–504번 트랙의 평균으로 best를 선택합니다. 학습 횟수는 모델별로 기록됩니다.

In [ ]:
import torch
from temporal_betappo import assert_action_mapping, train

torch.set_num_threads(2)
assert_action_mapping()
checkpoints = {}
for experts in EXPERT_COUNTS:
    root = RUN_BASE / f"experts{experts}_s{TRAIN_SEED}"
    root.mkdir(parents=True, exist_ok=True)
    checkpoints[experts] = (
        Path(EXISTING_CHECKPOINTS[experts]) if experts in EXISTING_CHECKPOINTS
        else train(root, experts, TRAIN_SEED, BUDGET, VALIDATION_SEEDS)
    )

/usr/local/lib/python3.13/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


25,595/300,000 raw frames | validation -47.8 | KL 0.0106
50,168/300,000 raw frames | validation -12.9 | KL 0.0127
75,767/300,000 raw frames | validation 242.5 | KL 0.0127
100,343/300,000 raw frames | validation 201.1 | KL 0.0067
125,942/300,000 raw frames | validation 192.7 | KL 0.0101
150,517/300,000 raw frames | validation 176.2 | KL 0.0106
175,093/300,000 raw frames | validation 359.7 | KL 0.0110
200,693/300,000 raw frames | validation 281.0 | KL 0.0349
225,269/300,000 raw frames | validation 440.2 | KL 0.0082
250,869/300,000 raw frames | validation 220.0 | KL 0.0162
275,445/300,000 raw frames | validation 435.6 | KL 0.0250
300,000/300,000 raw frames | validation 378.5 | KL 0.0402
25,598/300,000 raw frames | validation -76.5 | KL 0.0075
50,173/300,000 raw frames | validation -66.7 | KL 0.0168
75,773/300,000 raw frames | validation -71.2 | KL 0.0114
100,341/300,000 raw frames | validation -49.9 | KL 0.0116
125,940/300,000 raw frames | validation 223.0 | KL 0.0102
150,515/300,000 raw 

## 4. ONNX 검증과 동일 관측 행동 진단

- `official_scores.csv`는 공식 채점기로 계산한 원본 점수입니다.
- `action_diagnostics.csv`는 평균 주행의 매 결정에서 관측을 고정하고 Beta 행동 64개를
  뽑은 결과입니다. `mean_*`는 잠재변수 평균을 환경 행동으로 변환한 값입니다.
  `sample_mean_*`는 각 샘플을 환경 행동으로 변환한 뒤 평균한 값입니다.
  변환이 비선형이므로 두 값이 달라질 수 있습니다. `sample_both`는 0이어야 합니다.
- `diagnostic_scores.csv`는 같은 트랙 seed에서 평균 정책과 샘플링 정책으로 각각
  시작부터 주행한 결과입니다. 같은 물리 상태에서 비교한 값은 위 행동 분포 CSV에 있습니다.
- `recovery_events.csv`는 바퀴가 모두 도로를 놓친 뒤 100프레임 안에 재접촉,
  10프레임 연속 접촉, 새 타일 방문이 각각 발생했는지를 기록합니다.
  해결되지 않은 재접촉·재이탈은 같은 이벤트로 묶습니다. `curve`는 최근접 타일
  전방의 각도 차이로 분류한 분석용 표시입니다.

In [ ]:
from temporal_betappo import evaluate_and_diagnose, export_and_verify, load_checkpoint

for experts, checkpoint in checkpoints.items():
    root = RUN_BASE / f"experts{experts}_s{TRAIN_SEED}"
    model = load_checkpoint(checkpoint)
    if model.experts != experts:
        raise ValueError(f"Checkpoint expert count does not match {experts}")
    onnx_path = root / f"{TEAM_NAME}_TemporalBetaPPO_{experts}Expert.onnx"
    session, metadata = export_and_verify(model, onnx_path, seed=DIAGNOSTIC_SEEDS[0])
    evaluate_and_diagnose(model, root, session, metadata, EVAL_SEEDS,
                          DIAGNOSTIC_SEEDS, SAMPLE_REPEATS)
    print(root)

ONNX parity | 500 decisions | maximum error 1.91e-06
Official ONNX | 20 tracks | 249.33 ± 112.32
temporal_betappo_runs/20261107_152731/experts1_s42
ONNX parity | 500 decisions | maximum error 8.34e-07
Official ONNX | 20 tracks | 264.98 ± 117.85
temporal_betappo_runs/20261107_152731/experts2_s42


## 5. 평균 행동이 감속·복귀 입력을 약하게 만드는지 보기

In [ ]:
import pandas as pd
from IPython.display import display

for experts in checkpoints:
    root = RUN_BASE / f"experts{experts}_s{TRAIN_SEED}"
    print(f"Experts={experts}")
    actions = pd.read_csv(root / "action_diagnostics.csv")
    cols = ["mean_gas", "mean_brake", "sample_mean_gas", "sample_mean_brake",
            "sample_braking", "sample_left", "sample_right"] + [
                col for col in actions if col.startswith("gate_")]
    display(actions.groupby(["seed", "phase"])[cols].mean().round(3))
    display(pd.read_csv(root / "diagnostic_scores.csv"))
    if (root / "recovery_events.csv").exists():
        display(pd.read_csv(root / "recovery_events.csv"))

Experts=1


mean_gas  mean_brake  sample_mean_gas  sample_mean_brake  \
seed phase                                                             
1004 curve     0.363       0.000            0.376              0.028   
     edge      0.328       0.000            0.360              0.029   
     grass     0.266       0.003            0.313              0.051   
     road      0.350       0.009            0.379              0.030   
1007 curve     0.340       0.000            0.369              0.031   
     edge      0.295       0.000            0.335              0.038   
     grass     0.284       0.000            0.328              0.041   
     road      0.344       0.006            0.371              0.033   

            sample_braking  sample_left  sample_right  gate_0  
seed phase                                                     
1004 curve           0.121        0.452         0.520     1.0  
     edge            0.138        0.658         0.285     1.0  
     grass           0.190        0.560         0.381     1.0  
     road            0.111        0.513         0.423     1.0  
1007 curve           0.131        0.555         0.407     1.0  
     edge            0.156        0.706         0.254     1.0  
     grass           0.167        0.586         0.345     1.0  
     road            0.127        0.451         0.498     1.0

,seed,mode,replicate,score,raw_frames,exits,recontacts,stable_recoveries,progressed
0,1004,mean,0,138.255034,1000,3,1,1,1
1,1004,sample,1,379.865772,1000,5,3,2,3
2,1004,sample,2,732.214765,1000,3,3,3,3
3,1004,sample,3,584.563758,1000,7,7,6,5
4,1007,mean,0,427.508091,1000,4,3,3,3
5,1007,sample,1,243.042071,1000,1,0,0,0
6,1007,sample,2,764.077670,1000,7,7,6,7
7,1007,sample,3,327.184466,1000,4,2,2,2


,seed,mode,replicate,start,touch,stable,progress,tiles_at_exit,tiles_at_touch
0,1004,mean,0,130,3.0,12.0,3.0,30,31.0
1,1004,mean,0,150,NaN,NaN,NaN,34,NaN
2,1004,mean,0,652,NaN,NaN,NaN,71,NaN
3,1004,sample,1,153,12.0,21.0,12.0,26,27.0
4,1004,sample,1,185,98.0,NaN,98.0,30,31.0
5,1004,sample,1,379,NaN,NaN,NaN,52,NaN
6,1004,sample,1,769,2.0,11.0,2.0,70,71.0
7,1004,sample,1,991,NaN,NaN,NaN,143,NaN
8,1004,sample,2,139,4.0,13.0,4.0,29,30.0
9,1004,sample,2,433,23.0,32.0,23.0,114,115.0


Experts=2


mean_gas  mean_brake  sample_mean_gas  sample_mean_brake  \
seed phase                                                             
1004 curve     0.284       0.000            0.319              0.044   
     edge      0.348       0.000            0.379              0.029   
     grass     0.256       0.002            0.324              0.067   
     road      0.341       0.001            0.366              0.024   
1007 curve     0.340       0.000            0.372              0.025   
     edge      0.357       0.000            0.383              0.025   
     grass     0.250       0.000            0.318              0.072   
     road      0.350       0.002            0.374              0.022   

            sample_braking  sample_left  sample_right  gate_0  gate_1  
seed phase                                                             
1004 curve           0.172        0.398         0.540   0.285   0.715  
     edge            0.119        0.711         0.218   0.449   0.551  
     grass           0.223        0.658         0.286   0.480   0.520  
     road            0.109        0.464         0.464   0.309   0.691  
1007 curve           0.108        0.553         0.377   0.366   0.634  
     edge            0.104        0.826         0.109   0.492   0.508  
     grass           0.238        0.696         0.239   0.505   0.495  
     road            0.100        0.462         0.469   0.322   0.678

,seed,mode,replicate,score,raw_frames,exits,recontacts,stable_recoveries,progressed
0,1004,mean,0,339.597315,1000,3,1,1,1
1,1004,sample,1,131.543624,1000,4,3,3,2
2,1004,sample,2,279.194631,1000,3,1,1,1
3,1004,sample,3,671.812081,1000,1,0,0,0
4,1007,mean,0,243.042071,1000,1,0,0,0
5,1007,sample,1,291.585761,1000,3,2,2,2
6,1007,sample,2,589.320388,1000,1,1,0,1
7,1007,sample,3,298.058252,1000,2,0,0,0


,seed,mode,replicate,start,touch,stable,progress,tiles_at_exit,tiles_at_touch
0,1004,mean,0,132,1.0,10.0,1.0,30,32.0
1,1004,mean,0,150,NaN,NaN,NaN,36,NaN
2,1004,mean,0,650,NaN,NaN,NaN,131,NaN
3,1004,sample,1,146,2.0,11.0,2.0,30,32.0
4,1004,sample,1,243,NaN,NaN,NaN,42,NaN
5,1004,sample,1,732,16.0,25.0,NaN,51,51.0
6,1004,sample,1,963,27.0,36.0,27.0,66,67.0
7,1004,sample,2,147,2.0,11.0,2.0,30,32.0
8,1004,sample,2,167,NaN,NaN,NaN,35,NaN
9,1004,sample,2,384,NaN,NaN,NaN,46,NaN


In [ ]:
# python check_submission.py --onnx MyTeam.onnx
!python check_submission.py --onnx MyTeam.onnx --seeds 1000-1019

/usr/local/lib/python3.13/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
제출 파일 오류: FileNotFoundError: [Errno 2] No such file or directory: 'MyTeam.onnx'


https://chatgpt.com/g/g-p-6a7b1d1b48c88191b49dbf1d8e812db7-c-for-rl/c/6aeefd17-e704-83e8-8c3a-d6e312b10bab